In [ ]:
import os
import shutil
from tqdm import tqdm
import logging

import os
from collections import defaultdict

def dataset_statistics(download_path):
    """
    Prints dataset statistics after reorganization
    
    Structure expected:
    download_path/species/Good|Bad|Misclassified/images
    """

    stats = defaultdict(lambda: defaultdict(int))
    total_images = 0

    for species in os.listdir(download_path):
        species_path = os.path.join(download_path, species)

        if not os.path.isdir(species_path):
            continue

        for label in os.listdir(species_path):
            label_path = os.path.join(species_path, label)

            if not os.path.isdir(label_path):
                continue

            count = len([
                f for f in os.listdir(label_path)
                if os.path.isfile(os.path.join(label_path, f))
            ])

            stats[species][label] = count
            total_images += count

    # Print nicely
    print("\n📊 DATASET STATISTICS\n")
    print("-" * 60)
    print(f"{'Species':20} {'Good':10} {'Bad':10} {'Misclassified':15}")
    print("-" * 60)

    total_good = total_bad = total_mis = 0

    for species, labels in sorted(stats.items()):
        good = labels.get("Good", 0)
        bad = labels.get("Bad", 0)
        mis = labels.get("Misclassified", 0)

        total_good += good
        total_bad += bad
        total_mis += mis

        print(f"{species:20} {good:<10} {bad:<10} {mis:<15}")

    print("-" * 60)
    print(f"{'TOTAL':20} {total_good:<10} {total_bad:<10} {total_mis:<15}")
    print(f"\n📦 Total Images: {total_images}\n")
    
    
def reorganize_species_wise(
    download_path,
    dry_run=True,
    handle_misclassified=True,
    remove_empty=True,
    log_file="reorganization.log"
):
    """
    Reorganize dataset:
    
    FROM:
    download_path/date/species/Fresh-Stale/images
    
    TO:
    download_path/species/Good-Bad/images
    
    Parameters:
    ----------
    dry_run : bool
        If True, only shows what will happen (safe mode)
        
    handle_misclassified : bool
        Move misclassified images to separate folder
        
    remove_empty : bool
        Remove empty folders after moving
        
    log_file : str
        Log file name
    """

    # Setup logging
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(message)s"
    )

    print("\n🚀 Starting Reorganization...\n")

    dates = os.listdir(download_path)

    for date in tqdm(dates, desc="Processing Dates"):
        date_path = os.path.join(download_path, date)

        if not os.path.isdir(date_path):
            continue

        # Skip if already species folder
        if "-" not in date and date[0].isalpha():
            continue

        for species in os.listdir(date_path):

            species_path = os.path.join(date_path, species)

            if not os.path.isdir(species_path):
                continue

            for freshness in os.listdir(species_path):

                freshness_path = os.path.join(species_path, freshness)

                if not os.path.isdir(freshness_path):
                    continue

                # Create new directory
                new_species_path = os.path.join(download_path, species)
                new_label_path = os.path.join(new_species_path, freshness)

                if not dry_run:
                    os.makedirs(new_label_path, exist_ok=True)

                # Move images
                for image in os.listdir(freshness_path):

                    src = os.path.join(freshness_path, image)

                    # Handle Misclassified
                    if handle_misclassified and "Misclassified" in freshness_path:
                        mis_path = os.path.join(
                            download_path,
                            species,
                            "Misclassified"
                        )

                        dst = os.path.join(mis_path, image)

                        if not dry_run:
                            os.makedirs(mis_path, exist_ok=True)

                    else:
                        dst = os.path.join(new_label_path, image)

                    # Avoid overwrite
                    if os.path.exists(dst):
                        base, ext = os.path.splitext(image)
                        dst = os.path.join(
                            os.path.dirname(dst),
                            f"{base}_{date}{ext}"
                        )

                    # Dry run print
                    if dry_run:
                        print(f"[DRY RUN] Move: {src} -> {dst}")
                    else:
                        shutil.move(src, dst)
                        logging.info(f"Moved: {src} -> {dst}")

    # Remove empty folders
    if remove_empty and not dry_run:

        print("\n🧹 Removing empty folders...")

        for root, dirs, files in os.walk(download_path, topdown=False):
            if not os.listdir(root):
                os.rmdir(root)

    print("\n✅ Reorganization Completed!\n")

    if dry_run:
        print("⚠️ Dry run enabled — No files were moved")
    else:
        print("🎉 Files moved successfully")
        dataset_statistics(download_path)

In [ ]:
download_path = r"/content/drive/MyDrive/Sowmya /qZense Dataset/S3 Data/Daily Data"

reorganize_species_wise(
    download_path,
    dry_run=True
)